In [2]:
a = [2, 3, 4]
b = [1, 1, 0]

In [16]:
sum([a[i] * (b[i] == 1) for i in range(len(a))]) / sum([(b[i] == 1) for i in range(len(a))])

2.5

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import segmentation_models_pytorch as smp # ★ 忘れずインポート
from generator_modules import *

C:\Users\kohei\PycharmProjects\PythonProject\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
C:\Users\kohei\PycharmProjects\PythonProject\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have ha

In [3]:
encoder = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=2,
    classes=1  # (このc_outはダミー)
).encoder

In [5]:
encoder.out_channels

[2, 64, 64, 128, 256, 512]

In [3]:
decoder = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=2,
    classes=1  # (このc_outはダミー)
).decoder

In [4]:
decoder.out_channels

AttributeError: 'UnetDecoder' object has no attribute 'out_channels'

In [8]:
model = UNet_gemini()

In [9]:
model.out_channels

AttributeError: 'UNet_gemini' object has no attribute 'out_channels'

In [3]:
depth = 1
for i in range(depth - 1):
    print(i)

In [4]:
target=["mito", "ER"]
len(target)

1

In [3]:
a = (1, 2, 4, 4)
type(a)

tuple

In [ ]:
import wandb
import random

# 1. wandbを初期化
wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="kohei_tokyo-the-university-of-tokyo",
    # Set the wandb project where this run will be logged.
    project="Digital_Staining"
)

#wandb.define_metric("train/*", step_metric="epoch")
#wandb.define_metric("train/accuracy", step_metric="x")
#wandb.define_metric("val/*", step_metric="x")
# --- トレーニングループの例 ---
#for step in range(100):
# ダミーの計算
step=1
loss = 1.0 - (step * 0.01) + random.random() * 0.1
accuracy = (step * 0.01) + random.random() * 0.1
loss_val = (step * 0.1) + random.random() * 0.1
accuracy_val = (step * 0.1) + random.random() * 0.1
x = step * 10

# 2. wandb.log()で数値を記録
# キーがグラフ名、バリューがその時点での値になる
wandb.log({
    "train/train2/loss": loss,
    "train/train2/accuracy": accuracy,
    #"epoch": step
})
wandb.log({

    "val/loss": loss_val,
    "val/accuracy": accuracy_val,
    #"x": x
})

# 3. 実行を終了
wandb.finish()

print("Wandbへの記録が完了しました。")

In [ ]:
class DigitalStaining():
    def __init__(
            self,
            dir=None,  # train or test : 元データのpath, predict : 入力画像のpath
            main_dir=None,  # train or test : 前処理データ保存フォルダのpath, predict : なし
            original_dir=None,  # train or test : なし, predict : 入力画像のpath
            new_dir=None,  # train or test : なし, predict : 予測画像保存フォルダのpath
            name="Run",  # 名称
            group=None,
            produce_image=False,  # 一度前処理をしている場合はFalse
            train_folders=["train"],  # Trainデータのフォルダ名
            val_folders=["val"],  # Valデータのフォルダ名
            test_folders=["test"],  # Testデータのフォルダ名
            target=None, # "original_ER", "preprocess_ER", "original_mito", "preprocess_mito"
            # produce_images
            img_n=100,
            img_size=256,
            # gan
            n_epoch=10,
            discriminator="Patch4",  # Patch4, Patch3, Patch5, ResnetPatch, Resnet, or U_Net
            num_workers=4,  # GPUのメモリが足りない場合は小さくしてください
            in_chans=2,
            w_l1=50,
            w_ssim=1.0,
            w_dice=1.0,
            crop_size=256,
            stride=128,
            learning_rate_g=0.0002,
            learning_rate_d=0.0002,
            betas=(0.5, 0.999),
            images_to_use="both",
            device=torch.device(f'cuda:{torch.cuda.current_device()}' if torch.cuda.is_available() else 'cpu'),
            patches_per_epoch=200,
            val_epoch=1,
            batch_size=16,
            decoder_attention_type=None, # 'scse' or None
            val_crop=True,
            epoch_start_ema=50,
            encorder_name="resnet34", # "resnet34", "resnet50", or "efficientnet-b4"
            # predict
            test_id="lpips"
    ):
        self.dir = Path(dir)
        self.main_dir = main_dir
        self.original_dir = original_dir
        self.new_dir = new_dir
        self.name = name
        self.group = group
        self.produce_image = produce_image
        self.train_folders = train_folders
        self.val_folders = val_folders
        self.test_folders = test_folders
        self.img_n = img_n
        self.img_size = img_size
        self.n_epoch = n_epoch
        self.discriminator = discriminator
        self.num_workers = num_workers
        self.in_chans = in_chans
        self.w_l1 = w_l1
        self.w_ssim = w_ssim
        self.w_dice = w_dice
        self.img_size = img_size
        self.crop_size = crop_size
        self.stride = stride
        self.learning_rate_g = learning_rate_g
        self.learning_rate_d = learning_rate_d
        self.betas = betas
        self.images_to_use = images_to_use
        self.device = device
        self.patches_per_epoch = patches_per_epoch
        self.val_epoch = val_epoch
        self.batch_size = batch_size
        self.decoder_attention_type = decoder_attention_type
        self.val_crop = val_crop
        self.epoch_start_ema = epoch_start_ema
        self.encorder_name = encorder_name
        self.test_id = test_id
